# Whisper Arabic ASR Baseline
Run Whisper on a Common Voice Arabic subset and compute WER.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import re
import time
import numpy as np
import pandas as pd
import torch
import whisper

try:
    import miniaudio
    HAS_MINIAUDIO = True
except ImportError:
    HAS_MINIAUDIO = False

try:
    import torchaudio
    HAS_TORCHAUDIO = True
except ImportError:
    HAS_TORCHAUDIO = False

def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""

    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    text = re.sub(r"[^\w\s]", "", text)

    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"[يى]", "ي", text)
    text = re.sub(r"ة", "ه", text)

    text = re.sub(r"[^ا-ي\s]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
DATA_DIR = "../data/raw"
MODEL_SIZE = "medium"
LIMIT = 200

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
device

In [ ]:
model = whisper.load_model(MODEL_SIZE, device=device)

In [ ]:
clips_dir = os.path.join(DATA_DIR, "clips")
test_tsv = os.path.join(DATA_DIR, "test.tsv")

df = pd.read_csv(test_tsv, sep="\t")
df["sentence"] = df["sentence"].apply(clean_arabic_text)
df = df[df["sentence"].str.len() > 0].reset_index(drop=True)
if LIMIT is not None:
    df = df.head(int(LIMIT))

references, hypotheses, wers = [], [], []
start = time.time()

for i, row in df.iterrows():
    audio_path = os.path.join(clips_dir, row["path"])
    audio_np = load_audio_numpy(audio_path)
    if audio_np is None:
        continue

    result = model.transcribe(
        audio_np,
        language="ar",
        task="transcribe",
        fp16=False,
        beam_size=5,
        best_of=5,
        temperature=0.0,
        condition_on_previous_text=False,
    )

    hyp = clean_arabic_text(result["text"])
    wer = word_error_rate(row["sentence"], hyp)

    references.append(row["sentence"])
    hypotheses.append(hyp)
    wers.append(wer)

    if (i + 1) % 50 == 0 or (i + 1) == len(df):
        avg_so_far = sum(wers) / max(len(wers), 1)
        elapsed = time.time() - start
        print(f"[{i + 1}/{len(df)}] running WER = {avg_so_far:.4f} | elapsed = {elapsed:.1f}s")

whisper_wer = sum(wers) / max(len(wers), 1)
whisper_wer

In [ ]:
if len(df) == 0:
    raise RuntimeError("No test samples found.")

sample_audio = os.path.join(clips_dir, df["path"].iloc[0])
audio_np = load_audio_numpy(sample_audio)
result = model.transcribe(
    audio_np,
    language="ar",
    task="transcribe",
    fp16=False,
)
result["text"]